In [ ]:
#| default_exp bg

# Background sessions

> Named, managed tmux sessions: create-or-reuse by name, drive by sid

fastmux's handle API assumes you can hold a `Pane` between calls. A background terminal often outlives that: it is created in one tool call and driven from another, shared with a user who attaches by name, and referred to by nothing but that name. This module provides that functionality.

`start_session` creates-or-reuses a named session, and every verb takes a `sid` — a session name, a `%pane_id`, a `Session` or `Pane` handle, or `None` for the current pane.

In [ ]:
#| export
from fastcore.utils import *
from fastmux.core import *
from fastmux.core import _tmux
import uuid

In [ ]:
#| hide
from fastcore.test import *

## Managed sessions

A managed session differs from a bare `new_session` in the ways background work needs: it is found again by name instead of erroring as a duplicate; `remain` is set (with the dead-pane banner cleared) so a finished command's output and exit status stay inspectable; and the created pane's id is recorded in the `@fastmux_pane_id` session option, so later splits don't change which pane the sid addresses. `cmd` is started with `respawn-pane` only after those options are in place, since a fast-failing command could otherwise die before they land.

In [ ]:
#| export
SESSION_PREFIX = 'fastmux-'

def start_session(
    sid=None,    # Session name; default: a generated `fastmux-`-prefixed name
    cmd=None,    # Command to run (str or argv list); default: the user's shell
    cwd=None,    # Working directory
    env=None,    # Extra environment vars as a dict
    width=None,  # Terminal width in columns
    height=None, # Terminal height in rows
):
    "The session named `sid`, created detached (managed, with `remain` set) if it doesn't exist"
    if sid is None: sid = f'{SESSION_PREFIX}{uuid.uuid4().hex[:10]}'
    try: return tmux(sid)
    except TmuxError: pass
    s = new_session(name=sid, width=width, height=height, cwd=cwd, env=env, remain=True)
    p = Pane.fetch(s.id)
    _tmux('set-option','-t',s.id,'@fastmux_pane_id',p.id)
    _tmux('set-option','-w','-t',p.id,'remain-on-exit-format','')
    if cmd is not None:
        args = ['respawn-pane','-k','-t',p.id]
        if cwd is not None: args += ['-c',os.fspath(cwd)]
        args += [x for k,v in (env or {}).items() for x in ('-e',f'{k}={v}')]
        args += [cmd] if isinstance(cmd,str) else list(cmd)
        _tmux(*args)
    return s

In [ ]:
s = start_session('fastmux-test-bg', width=60, height=8)
test_eq(s.name, 'fastmux-test-bg')
test_eq(start_session('fastmux-test-bg').id, s.id)
s

## Addressing by sid

`pane` turns any sid form into the `Pane` it addresses, and is the bridge to the full handle API: everything this module doesn't wrap (`wait`, `search`, splitting, transcript slicing) is a method on its result. For a session name, the managed pane recorded at creation wins over the currently-active one:

In [ ]:
#| export
def pane(sid=None):
    "The pane `sid` addresses: a session name (its managed pane), a `%pane_id`, a `Session` or `Pane` handle, or None for the current pane"
    if isinstance(sid, Pane): return sid
    if isinstance(sid, Session): sid = sid.name
    if sid is None: return current_pane()
    if sid.startswith('%'): return Pane.fetch(sid)
    _tmux('has-session','-t',f'={sid}')
    pid = _tmux('show-options','-qv','-t',sid,'@fastmux_pane_id')
    return Pane.fetch(pid or sid)

In [ ]:
p = pane('fastmux-test-bg')
test_eq(p.id, pane(s).id)
test_eq(p.id, pane(p.id).id)

Splitting the managed session doesn't move the sid's target:

In [ ]:
q = p.bsplit(size=3)
test_eq(pane('fastmux-test-bg').id, p.id)
q.kill()

## Driving by sid

The verbs mirror `Pane`'s, with a sid in front: each sends (or just waits), polls until the pane differs from its last-seen state, and returns a `Capture`. Output that arrived between calls satisfies the next `poll` immediately.

In [ ]:
#| export
def send(sid=None, chars='', wait_ms=0, interval_ms=50, lines=80):
    "Paste `chars` into `sid`'s pane literally, then `poll`"
    return pane(sid).send(chars, wait_ms, interval_ms, lines)

def send_keys(sid=None, *keys, wait_ms=0, interval_ms=50, lines=80):
    "Send tmux key names to `sid`'s pane, then `poll`"
    return pane(sid).send_keys(*keys, wait_ms=wait_ms, interval_ms=interval_ms, lines=lines)

def interrupt(sid=None, wait_ms=0, interval_ms=50, lines=80):
    "Send `Ctrl-C` to `sid`'s pane, then `poll`"
    return pane(sid).interrupt(wait_ms, interval_ms, lines)

def poll(sid=None, wait_ms=0, interval_ms=50, lines=80):
    "Wait for `sid`'s pane to differ from its last-seen state, then capture the last `lines`"
    return pane(sid).poll(wait_ms, interval_ms, lines)

def display(sid=None, lines=80):
    "Capture the last `lines` of `sid`'s pane"
    return pane(sid).display(lines)

In [ ]:
c = send('fastmux-test-bg', 'echo $((6*7)) apples\n', wait_ms=2000)
assert '42 apples' in c.text
c

## Lifecycle

`close` kills the whole session owning the sid's pane — including when the sid is a `%pane_id` — and `managed_sessions` lists every session `start_session` created, whatever it was named:

In [ ]:
#| export
def close(sid=None):
    "Kill the session owning `sid`'s pane"
    Session.fetch(pane(sid).session).kill()

def managed_sessions():
    "Sessions created by `start_session`"
    return Sessions(s for s in tmux() if _tmux('show-options','-qv','-t',s.id,'@fastmux_pane_id'))

In [ ]:
test_eq('fastmux-test-bg' in managed_sessions().attrgot('name'), True)
close('fastmux-test-bg')
with expect_fail(TmuxError): pane('fastmux-test-bg')

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()